In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

TRAIN_FEATS_CSV = ""
TEST_FEATS_CSV = ""
TRAIN_LABELS_XLSX = ""
TEST_LABELS_XLSX = ""

## Train

In [ ]:
df = pd.read_csv(TRAIN_FEATS_CSV)
labels_df = pd.read_excel(TRAIN_LABELS_XLSX)[:len(df)]
data = df.merge(labels_df, on="id", how="inner")


train_df, test_df = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["label"],
)

X_train = train_df.filter(like="feat_")
y_train = train_df["label"]

X_test = test_df.filter(like="feat_")
y_test = test_df["label"]

test_ids = test_df["id"]

# Uncomment if you want to remove features
#to_drop = ['feat_56', 'feat_84', 'feat_2', 'feat_67', 'feat_7', 'feat_39', 'feat_43', 'feat_52', 'feat_49']
#X_train = X_train.drop(columns=to_drop)
#X_test  = X_test.drop(columns=to_drop)

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced", 
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

rf_wrong = test_df.loc[y_pred != y_test, ["id", "label"]].copy()
rf_wrong["predicted_label"] = y_pred[y_pred != y_test]

# Uncomment to see misclassified samples
#print(rf_wrong)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## ImpPrune

In [ ]:
rf_importances = pd.Series(rf.feature_importances_, index=X_train.columns)

maxf1 = 0
bestK = 0 
bestlen = 0
drop_list = []
for K in range(0, 89, 1):
    low_rf = rf_importances.sort_values().head(K).index
    to_drop = low_rf
    print(f"Dropping {len(to_drop)} features")

    X_train_r = X_train.drop(columns=to_drop)
    X_test_r  = X_test.drop(columns=to_drop)

    rf2 = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
    )
    rf2.fit(X_train_r, y_train)
    rf2_pred = rf2.predict(X_test_r)

    f1 = f1_score(y_test, rf2_pred, average="macro")
    if f1 >= maxf1:
        maxf1 = f1
        bestK = K
        bestlen = len(to_drop)
        drop_list = to_drop.tolist()

    print("RF reduced Accuracy:", accuracy_score(y_test, rf2_pred))
    print("RF reduced Macro-F1:", f1_score(y_test, rf2_pred, average="macro"))
print(f"Best K: {bestK} with Macro-F1: {maxf1:.4f}")
print(f"Dropped {bestlen} features: {drop_list}")


## F1Prune

In [ ]:
best = (set(), 0)
to_check = [best] 

while to_check:
    current, curr_f1 = to_check.pop(0)
    print(f"Checking {len(current)} features dropped...")

    X_train_r = X_train.drop(columns=current)
    X_test_r  = X_test.drop(columns=current)

    rf2 = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
    )
    rf2.fit(X_train_r, y_train)
    rf2_pred = rf2.predict(X_test_r)

    f1 = f1_score(y_test, rf2_pred, average="macro")
    if f1 > best[1]:
        for feat in X_train.columns:
            if feat not in current:
                new_set = current.union({feat})
                to_check.append((new_set, f1))
        best = (current, f1)
        print(f"New best: {len(current)} features dropped with Macro-F1: {f1:.4f}")

    print(len(to_check), "sets left to check...")

print(best)

## Run Test

In [ ]:
df = pd.read_csv(TEST_FEATS_CSV)

X_test = df.filter(like="feat_")

X_test  = X_test.drop(columns=to_drop)

y_pred = rf.predict(X_test)

y_true = pd.read_excel(TEST_LABELS_XLSX)["label"]
print(len(y_true), len(y_pred))
print(classification_report(y_true, y_pred))

### Selected Features for Removal

In [ ]:
imp_all = ['feat_12', 'feat_76', 'feat_28', 'feat_27', 'feat_84', 'feat_38', 'feat_26', 'feat_15', 'feat_79', 'feat_85', 'feat_22', 'feat_17', 'feat_75', 'feat_72', 'feat_82', 'feat_83', 'feat_10', 'feat_36', 'feat_57', 'feat_3', 'feat_19', 'feat_80', 'feat_20', 'feat_9', 'feat_1', 'feat_13', 'feat_25', 'feat_4', 'feat_11', 'feat_21', 'feat_63', 'feat_51', 'feat_37', 'feat_88', 'feat_23', 'feat_24', 'feat_49', 'feat_5', 'feat_77', 'feat_86', 'feat_56', 'feat_32', 'feat_59', 'feat_44', 'feat_71', 'feat_61', 'feat_14', 'feat_34', 'feat_2', 'feat_43', 'feat_18', 'feat_68', 'feat_78', 'feat_30', 'feat_73', 'feat_62', 'feat_74', 'feat_55', 'feat_33', 'feat_48', 'feat_40', 'feat_31', 'feat_66', 'feat_54', 'feat_67', 'feat_58', 'feat_6', 'feat_16', 'feat_29', 'feat_52', 'feat_89', 'feat_69', 'feat_45', 'feat_60', 'feat_47', 'feat_53', 'feat_81', 'feat_70', 'feat_39', 'feat_64', 'feat_41', 'feat_46', 'feat_65', 'feat_87', 'feat_42']
f1_all = ['feat_56', 'feat_84', 'feat_2', 'feat_67', 'feat_7', 'feat_39', 'feat_43', 'feat_52', 'feat_49']

imp_eng = ['feat_12', 'feat_28', 'feat_84', 'feat_85', 'feat_26', 'feat_27', 'feat_76', 'feat_79', 'feat_49', 'feat_38', 'feat_88', 'feat_13', 'feat_82', 'feat_15', 'feat_67', 'feat_80', 'feat_11', 'feat_72', 'feat_22', 'feat_17']
f1_eng = ['feat_28', 'feat_55']

imp_chi = ['feat_12', 'feat_27', 'feat_76', 'feat_38', 'feat_26', 'feat_79', 'feat_28', 'feat_17', 'feat_84', 'feat_82', 'feat_83', 'feat_85', 'feat_13', 'feat_10', 'feat_72', 'feat_3', 'feat_15', 'feat_36', 'feat_80', 'feat_75', 'feat_71', 'feat_57', 'feat_4', 'feat_44', 'feat_1', 'feat_25', 'feat_77', 'feat_63', 'feat_43', 'feat_61', 'feat_19', 'feat_5', 'feat_88', 'feat_74', 'feat_20', 'feat_24', 'feat_86', 'feat_22', 'feat_9', 'feat_11', 'feat_73', 'feat_56', 'feat_59', 'feat_2', 'feat_6', 'feat_51', 'feat_49', 'feat_23', 'feat_16', 'feat_64', 'feat_81']
f1_chi = ['feat_55']

imp_hin = ['feat_12', 'feat_76', 'feat_27', 'feat_28', 'feat_75', 'feat_15', 'feat_17', 'feat_57', 'feat_3', 'feat_44', 'feat_22', 'feat_61', 'feat_84', 'feat_1', 'feat_77', 'feat_38', 'feat_85', 'feat_36', 'feat_83', 'feat_4', 'feat_11', 'feat_26', 'feat_72', 'feat_82', 'feat_19', 'feat_5', 'feat_51', 'feat_62', 'feat_20', 'feat_32', 'feat_9', 'feat_79', 'feat_37', 'feat_80', 'feat_63', 'feat_10', 'feat_88', 'feat_49', 'feat_73', 'feat_13', 'feat_30', 'feat_86', 'feat_21', 'feat_60', 'feat_54', 'feat_25', 'feat_43', 'feat_24', 'feat_56', 'feat_14', 'feat_68', 'feat_74', 'feat_23', 'feat_52', 'feat_48', 'feat_70']
f1_hin = ['feat_66', 'feat_50', 'feat_69', 'feat_88']

In [ ]:
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if len(a | b) > 0 else 0

# pairwise similarities
j_eng_chi = jaccard(f1_eng, f1_chi)
j_eng_hin = jaccard(f1_eng, f1_hin)
j_chi_hin = jaccard(f1_chi, f1_hin)

print("ENG-CHI:", round(j_eng_chi, 3))
print("ENG-HIN:", round(j_eng_hin, 3))
print("CHI-HIN:", round(j_chi_hin, 3))

def jaccard_three(a, b, c):
    a, b, c = set(a), set(b), set(c)
    intersection = a & b & c
    union = a | b | c
    return len(intersection) / len(union) if len(union) > 0 else 0

j_all = jaccard_three(f1_eng, f1_chi, f1_hin)
print("ALL THREE:", round(j_all, 3))

def intersection(a, b):
    return set(a) & set(b)

int_eng_chi = intersection(f1_eng, f1_chi)
int_eng_hin = intersection(f1_eng, f1_hin)
int_chi_hin = intersection(f1_chi, f1_hin)

print("ENG-CHI:", len(int_eng_chi))
print("ENG-HIN:", len(int_eng_hin))
print("CHI-HIN:", len(int_chi_hin))
